# Three ML Models Competition

This notebook trains and compares **three ML policies** for tote sequencing:

- `linear_logreg` (linear logistic model)
- `neural_net` (1 hidden-layer NN)
- `tree_boost` (tree-based gradient-boosted stumps)

All three optimize the same composite objective:

`objective_score = total_time - OPTIONALITY_LAMBDA * optionality_score`

Exact optimization is used only as supervision/benchmark to compute gaps.


In [ ]:
import csv
import math
import random
from pathlib import Path

RUN_ID = "all"  # None, int, or "all"
TRAIN_FRACTION = 0.35
MAX_TRAIN_RUNS = 180
RNG_SEED = 42

PLACE_TIME = 1.75
TOTE_SWITCH_TIME = 4.0
BIN_SWITCH_TIME = 0.75

OPTIONALITY_LAMBDA = 0.35
OPT_EDGE_WEIGHT = 1.0
OPT_BRANCH_WEIGHT = 0.6
OPT_RARE_WEIGHT = 0.4

random.seed(RNG_SEED)


def _resolve_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return paths[0]


def _get_input_bases(run_id):
    if run_id == "all":
        runs_root = _resolve_existing([Path("inputs/runs"), Path("../inputs/runs")])
        if not runs_root.exists():
            return []
        return sorted([p for p in runs_root.iterdir() if p.is_dir() and p.name.startswith("run_")])
    if run_id is None:
        return [_resolve_existing([Path("inputs"), Path("../inputs")])]
    run_name = f"run_{int(run_id):04d}"
    return [_resolve_existing([Path("inputs/runs") / run_name, Path("../inputs/runs") / run_name])]


OUT = _resolve_existing([Path("outputs"), Path("../outputs")])
OUT.mkdir(parents=True, exist_ok=True)


def _coerce_int(v):
    s = v.strip()
    if s == "":
        return None
    try:
        return int(float(s))
    except ValueError:
        return None


def _read_rows(p):
    with p.open("r", newline="") as f:
        return list(csv.reader(f))


def load_blocks(base):
    input_itemtypes = base / "order_itemtypes.csv"
    input_quantities = base / "order_quantities.csv"
    input_totes = base / "orders_totes.csv"

    ir = _read_rows(input_itemtypes)
    qr = _read_rows(input_quantities)
    tr = _read_rows(input_totes)
    n = max(len(ir), len(qr), len(tr))

    tote_bins = {}
    for i in range(n):
        row_i = ir[i] if i < len(ir) else []
        row_q = qr[i] if i < len(qr) else []
        row_t = tr[i] if i < len(tr) else []
        w = max(len(row_i), len(row_q), len(row_t))
        for j in range(w):
            qty = _coerce_int(row_q[j]) if j < len(row_q) else None
            tote = _coerce_int(row_t[j]) if j < len(row_t) else None
            if qty is None or tote is None or qty <= 0:
                continue
            tote_bins.setdefault(tote, []).extend([i + 1] * qty)

    blocks = {}
    for tote, bins in tote_bins.items():
        b = sorted(bins)
        blocks[tote] = {
            "first_bin": b[0],
            "last_bin": b[-1],
            "units": len(b),
            "internal_switches": sum(1 for k in range(1, len(b)) if b[k] != b[k - 1]),
        }
    return blocks


def transition_cost(prev_tote, prev_last_bin, tote, blocks):
    if prev_tote is None:
        return 0.0
    c = TOTE_SWITCH_TIME
    if prev_last_bin != blocks[tote]["first_bin"]:
        c += BIN_SWITCH_TIME
    return c


def block_cost(tote, blocks):
    b = blocks[tote]
    return b["units"] * PLACE_TIME + b["internal_switches"] * BIN_SWITCH_TIME


def build_optionality_terms(blocks):
    totes = sorted(blocks.keys())
    first_bins = {t: blocks[t]["first_bin"] for t in totes}
    last_bins = {t: blocks[t]["last_bin"] for t in totes}

    compat = set()
    branch = {t: 0 for t in totes}
    for i in totes:
        for j in totes:
            if i == j:
                continue
            if last_bins[i] == first_bins[j]:
                compat.add((i, j))
                branch[i] += 1

    bin_freq = {}
    for t in totes:
        b = first_bins[t]
        bin_freq[b] = bin_freq.get(b, 0) + 1

    node_coeff = {}
    for t in totes:
        rarity = 1.0 / bin_freq[first_bins[t]]
        node_coeff[t] = OPT_BRANCH_WEIGHT * branch[t] - OPT_RARE_WEIGHT * rarity

    return {"compat": compat, "node_coeff": node_coeff}


def sequence_metrics(seq, blocks, terms):
    n = len(seq)
    total_time = 0.0
    optionality = 0.0
    prev_tote = None
    prev_last_bin = None
    for pos, tote in enumerate(seq, start=1):
        total_time += transition_cost(prev_tote, prev_last_bin, tote, blocks) + block_cost(tote, blocks)
        optionality += terms["node_coeff"][tote] * (n + 1 - pos)
        if prev_tote is not None and (prev_tote, tote) in terms["compat"]:
            optionality += OPT_EDGE_WEIGHT
        prev_tote = tote
        prev_last_bin = blocks[tote]["last_bin"]
    objective = total_time - OPTIONALITY_LAMBDA * optionality
    return total_time, optionality, objective


def solve_exact(blocks, terms):
    totes = sorted(blocks.keys())
    n = len(totes)
    if n == 0:
        return []

    full = 1 << n
    inf = 10 ** 30
    dp = [[inf] * n for _ in range(full)]
    parent = [[-1] * n for _ in range(full)]

    for j, tote in enumerate(totes):
        pos_factor = float(n)
        opt = terms["node_coeff"][tote] * pos_factor
        dp[1 << j][j] = block_cost(tote, blocks) - OPTIONALITY_LAMBDA * opt

    popcount = [0] * full
    for m in range(1, full):
        popcount[m] = popcount[m >> 1] + (m & 1)

    for mask in range(full):
        used = popcount[mask]
        if used == 0:
            continue
        for j in range(n):
            cur = dp[mask][j]
            if cur >= inf:
                continue
            last_tote = totes[j]
            for k in range(n):
                if mask & (1 << k):
                    continue
                next_tote = totes[k]
                pos_factor = float(n - used)
                opt = terms["node_coeff"][next_tote] * pos_factor
                if (last_tote, next_tote) in terms["compat"]:
                    opt += OPT_EDGE_WEIGHT
                cand = cur + block_cost(next_tote, blocks) + transition_cost(last_tote, blocks[last_tote]["last_bin"], next_tote, blocks) - OPTIONALITY_LAMBDA * opt
                nm = mask | (1 << k)
                if cand < dp[nm][k]:
                    dp[nm][k] = cand
                    parent[nm][k] = j

    end = min(range(n), key=lambda j: dp[full - 1][j])
    seq_idx = []
    mask = full - 1
    cur = end
    while cur != -1:
        seq_idx.append(cur)
        p = parent[mask][cur]
        mask ^= (1 << cur)
        cur = p
    seq_idx.reverse()
    return [totes[i] for i in seq_idx]


In [ ]:
def _sigmoid(x):
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def feature_vector(prev_tote, prev_last_bin, tote, pos, n, blocks, terms, remaining_size):
    time_inc = transition_cost(prev_tote, prev_last_bin, tote, blocks) + block_cost(tote, blocks)
    opt_inc = terms["node_coeff"][tote] * (n + 1 - pos)
    compat = 1.0 if (prev_tote is not None and (prev_tote, tote) in terms["compat"]) else 0.0
    if compat > 0:
        opt_inc += OPT_EDGE_WEIGHT
    return [
        1.0,
        time_inc,
        opt_inc,
        compat,
        float(blocks[tote]["units"]),
        float(blocks[tote]["internal_switches"]),
        float(pos) / max(1.0, float(n)),
        float(remaining_size),
    ]


def build_training_data(train_runs):
    X = []
    y = []
    for base in train_runs:
        blocks = load_blocks(base)
        if not blocks:
            continue
        terms = build_optionality_terms(blocks)
        exact_seq = solve_exact(blocks, terms)
        remaining = set(exact_seq)
        prev_tote = None
        prev_last_bin = None
        n = len(exact_seq)

        for pos, chosen in enumerate(exact_seq, start=1):
            for tote in sorted(remaining):
                X.append(feature_vector(prev_tote, prev_last_bin, tote, pos, n, blocks, terms, len(remaining)))
                y.append(1.0 if tote == chosen else 0.0)
            remaining.remove(chosen)
            prev_tote = chosen
            prev_last_bin = blocks[chosen]["last_bin"]
    return X, y


def standardize_fit(X):
    n = len(X)
    d = len(X[0])
    means = [0.0] * d
    stds = [1.0] * d
    for j in range(1, d):
        col = [row[j] for row in X]
        mu = sum(col) / n
        var = sum((v - mu) ** 2 for v in col) / n
        sd = math.sqrt(var) if var > 1e-12 else 1.0
        means[j] = mu
        stds[j] = sd
    return means, stds


def standardize_apply(X, means, stds):
    out = []
    for row in X:
        r = row[:]
        for j in range(1, len(r)):
            r[j] = (r[j] - means[j]) / stds[j]
        out.append(r)
    return out


In [ ]:
# --- ML model 1: Linear logistic regression ---
def fit_linear_logreg(X, y, lr=0.03, epochs=220, l2=1e-4):
    means, stds = standardize_fit(X)
    Xn = standardize_apply(X, means, stds)
    m = len(Xn)
    d = len(Xn[0])
    w = [0.0] * d
    for _ in range(epochs):
        g = [0.0] * d
        for i in range(m):
            z = sum(w[j] * Xn[i][j] for j in range(d))
            p = _sigmoid(z)
            err = p - y[i]
            for j in range(d):
                g[j] += err * Xn[i][j]
        for j in range(d):
            g[j] = g[j] / m + l2 * w[j]
            w[j] -= lr * g[j]
    return {"name": "linear_logreg", "w": w, "means": means, "stds": stds}


def predict_linear(model, x):
    r = x[:]
    for j in range(1, len(r)):
        r[j] = (r[j] - model["means"][j]) / model["stds"][j]
    z = sum(model["w"][j] * r[j] for j in range(len(r)))
    return _sigmoid(z)


# --- ML model 2: 1-hidden-layer neural network ---
def fit_neural_net(X, y, hidden=12, lr=0.02, epochs=120):
    means, stds = standardize_fit(X)
    Xn = standardize_apply(X, means, stds)
    m = len(Xn)
    d = len(Xn[0])

    rnd = random.Random(123)
    W1 = [[(rnd.random() - 0.5) * 0.2 for _ in range(hidden)] for _ in range(d)]
    b1 = [0.0] * hidden
    W2 = [(rnd.random() - 0.5) * 0.2 for _ in range(hidden)]
    b2 = 0.0

    for _ in range(epochs):
        gW1 = [[0.0] * hidden for _ in range(d)]
        gb1 = [0.0] * hidden
        gW2 = [0.0] * hidden
        gb2 = 0.0

        for i in range(m):
            x = Xn[i]
            h = [0.0] * hidden
            for k in range(hidden):
                a = b1[k] + sum(x[j] * W1[j][k] for j in range(d))
                h[k] = math.tanh(a)
            z = b2 + sum(h[k] * W2[k] for k in range(hidden))
            p = _sigmoid(z)
            dz = p - y[i]

            for k in range(hidden):
                gW2[k] += dz * h[k]
            gb2 += dz

            for k in range(hidden):
                dh = dz * W2[k]
                da = dh * (1.0 - h[k] * h[k])
                gb1[k] += da
                for j in range(d):
                    gW1[j][k] += da * x[j]

        inv_m = 1.0 / m
        for j in range(d):
            for k in range(hidden):
                W1[j][k] -= lr * gW1[j][k] * inv_m
        for k in range(hidden):
            b1[k] -= lr * gb1[k] * inv_m
            W2[k] -= lr * gW2[k] * inv_m
        b2 -= lr * gb2 * inv_m

    return {
        "name": "neural_net",
        "means": means,
        "stds": stds,
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2,
    }


def predict_nn(model, x):
    r = x[:]
    for j in range(1, len(r)):
        r[j] = (r[j] - model["means"][j]) / model["stds"][j]
    hidden = len(model["b1"])
    h = [0.0] * hidden
    for k in range(hidden):
        a = model["b1"][k] + sum(r[j] * model["W1"][j][k] for j in range(len(r)))
        h[k] = math.tanh(a)
    z = model["b2"] + sum(h[k] * model["W2"][k] for k in range(hidden))
    return _sigmoid(z)


In [ ]:
# --- ML model 3: Tree-based gradient-boosted stumps ---
def fit_best_stump(X, residual):
    n = len(X)
    d = len(X[0])
    best = None
    for j in range(1, d):
        vals = sorted(set(row[j] for row in X))
        if len(vals) <= 1:
            continue
        k = min(12, len(vals) - 1)
        idxs = [int((t + 1) * (len(vals) - 1) / (k + 1)) for t in range(k)]
        thresholds = sorted(set(vals[idx] for idx in idxs))
        for th in thresholds:
            left = [i for i in range(n) if X[i][j] <= th]
            right = [i for i in range(n) if X[i][j] > th]
            if not left or not right:
                continue
            lval = sum(residual[i] for i in left) / len(left)
            rval = sum(residual[i] for i in right) / len(right)
            sse = sum((residual[i] - lval) ** 2 for i in left) + sum((residual[i] - rval) ** 2 for i in right)
            cand = {"feature": j, "threshold": th, "left": lval, "right": rval, "sse": sse}
            if best is None or cand["sse"] < best["sse"]:
                best = cand
    return best


def fit_tree_boost(X, y, n_estimators=35, lr=0.25):
    means, stds = standardize_fit(X)
    Xn = standardize_apply(X, means, stds)
    base = sum(y) / len(y)
    pred = [base] * len(y)
    stumps = []
    for _ in range(n_estimators):
        residual = [y[i] - pred[i] for i in range(len(y))]
        stump = fit_best_stump(Xn, residual)
        if stump is None:
            break
        stumps.append(stump)
        f = stump["feature"]
        th = stump["threshold"]
        for i in range(len(y)):
            update = stump["left"] if Xn[i][f] <= th else stump["right"]
            pred[i] += lr * update
    return {
        "name": "tree_boost",
        "means": means,
        "stds": stds,
        "base": base,
        "stumps": stumps,
        "lr": lr,
    }


def predict_tree_boost(model, x):
    r = x[:]
    for j in range(1, len(r)):
        r[j] = (r[j] - model["means"][j]) / model["stds"][j]
    s = model["base"]
    for stump in model["stumps"]:
        v = stump["left"] if r[stump["feature"]] <= stump["threshold"] else stump["right"]
        s += model["lr"] * v
    return max(0.0, min(1.0, s))


def build_policy(model_name, model):
    if model_name == "linear_logreg":
        return lambda x: predict_linear(model, x)
    if model_name == "neural_net":
        return lambda x: predict_nn(model, x)
    if model_name == "tree_boost":
        return lambda x: predict_tree_boost(model, x)
    raise ValueError("unknown model")


def solve_ml_policy(blocks, terms, score_fn):
    remaining = set(blocks.keys())
    seq = []
    prev_tote = None
    prev_last_bin = None
    n = len(remaining)
    while remaining:
        pos = len(seq) + 1
        best_tote = None
        best_key = None
        for tote in sorted(remaining):
            fv = feature_vector(prev_tote, prev_last_bin, tote, pos, n, blocks, terms, len(remaining))
            p = score_fn(fv)
            time_inc = transition_cost(prev_tote, prev_last_bin, tote, blocks) + block_cost(tote, blocks)
            opt_inc = terms["node_coeff"][tote] * (n + 1 - pos)
            if prev_tote is not None and (prev_tote, tote) in terms["compat"]:
                opt_inc += OPT_EDGE_WEIGHT
            obj_inc = time_inc - OPTIONALITY_LAMBDA * opt_inc
            key = (-p, obj_inc, tote)
            if best_key is None or key < best_key:
                best_key = key
                best_tote = tote
        seq.append(best_tote)
        remaining.remove(best_tote)
        prev_tote = best_tote
        prev_last_bin = blocks[best_tote]["last_bin"]
    return seq


def local_improve(seq, blocks, terms, max_passes=2):
    best = seq[:]
    _, _, best_obj = sequence_metrics(best, blocks, terms)
    n = len(best)
    improved = True
    passes = 0
    while improved and passes < max_passes:
        improved = False
        passes += 1
        for i in range(n):
            for j in range(i + 1, n):
                cand = best[:]
                cand[i], cand[j] = cand[j], cand[i]
                _, _, obj = sequence_metrics(cand, blocks, terms)
                if obj + 1e-9 < best_obj:
                    best = cand
                    best_obj = obj
                    improved = True
    return best


In [ ]:
bases = _get_input_bases(RUN_ID)
if not bases:
    raise RuntimeError("No input runs found. Generate run folders first.")

train_n = min(MAX_TRAIN_RUNS, max(1, int(len(bases) * TRAIN_FRACTION)))
train_runs = bases[:train_n]
eval_runs = bases

X, y = build_training_data(train_runs)
if not X:
    raise RuntimeError("Empty training data.")

model_linear = fit_linear_logreg(X, y)
model_nn = fit_neural_net(X, y)
model_tree = fit_tree_boost(X, y)

score_linear = build_policy("linear_logreg", model_linear)
score_nn = build_policy("neural_net", model_nn)
score_tree = build_policy("tree_boost", model_tree)

rows = []
wins = {"linear_logreg": 0, "neural_net": 0, "tree_boost": 0}

for base in eval_runs:
    blocks = load_blocks(base)
    if not blocks:
        continue
    terms = build_optionality_terms(blocks)

    seq_exact = solve_exact(blocks, terms)
    _, _, obj_exact = sequence_metrics(seq_exact, blocks, terms)

    seq_l = local_improve(solve_ml_policy(blocks, terms, score_linear), blocks, terms)
    _, _, obj_l = sequence_metrics(seq_l, blocks, terms)

    seq_n = local_improve(solve_ml_policy(blocks, terms, score_nn), blocks, terms)
    _, _, obj_n = sequence_metrics(seq_n, blocks, terms)

    seq_t = local_improve(solve_ml_policy(blocks, terms, score_tree), blocks, terms)
    _, _, obj_t = sequence_metrics(seq_t, blocks, terms)

    ml_vals = {
        "linear_logreg": obj_l,
        "neural_net": obj_n,
        "tree_boost": obj_t,
    }
    best_ml = min(ml_vals, key=ml_vals.get)
    wins[best_ml] += 1

    rows.append({
        "run_name": base.name,
        "exact_objective": obj_exact,
        "linear_logreg_objective": obj_l,
        "neural_net_objective": obj_n,
        "tree_boost_objective": obj_t,
        "linear_gap_vs_exact": obj_l - obj_exact,
        "nn_gap_vs_exact": obj_n - obj_exact,
        "tree_gap_vs_exact": obj_t - obj_exact,
        "best_ml_model": best_ml,
    })

per_run = OUT / "ml_models_per_run.csv"
with per_run.open("w", newline="") as f:
    w = csv.DictWriter(
        f,
        fieldnames=[
            "run_name",
            "exact_objective",
            "linear_logreg_objective",
            "neural_net_objective",
            "tree_boost_objective",
            "linear_gap_vs_exact",
            "nn_gap_vs_exact",
            "tree_gap_vs_exact",
            "best_ml_model",
        ],
    )
    w.writeheader()
    w.writerows(rows)

def _avg(arr):
    return (sum(arr) / len(arr)) if arr else float("nan")

summary = [
    {
        "model": "linear_logreg",
        "n_runs": len(rows),
        "mean_objective": _avg([r["linear_logreg_objective"] for r in rows]),
        "mean_gap_vs_exact": _avg([r["linear_gap_vs_exact"] for r in rows]),
        "win_count": wins["linear_logreg"],
    },
    {
        "model": "neural_net",
        "n_runs": len(rows),
        "mean_objective": _avg([r["neural_net_objective"] for r in rows]),
        "mean_gap_vs_exact": _avg([r["nn_gap_vs_exact"] for r in rows]),
        "win_count": wins["neural_net"],
    },
    {
        "model": "tree_boost",
        "n_runs": len(rows),
        "mean_objective": _avg([r["tree_boost_objective"] for r in rows]),
        "mean_gap_vs_exact": _avg([r["tree_gap_vs_exact"] for r in rows]),
        "win_count": wins["tree_boost"],
    },
]

summary_path = OUT / "ml_models_summary.csv"
with summary_path.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["model", "n_runs", "mean_objective", "mean_gap_vs_exact", "win_count"])
    w.writeheader()
    w.writerows(summary)

print(f"Trained on {len(train_runs)} runs using exact labels.")
print(f"Wrote: {per_run}")
print(f"Wrote: {summary_path}")
print("\nML model ranking (lower mean objective is better):")
for r in sorted(summary, key=lambda z: z["mean_objective"]):
    print(r)
